In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import os
import random

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

# Load BBC News Summary data
base_dir = "BBC News Summary"
articles_dir = os.path.join(base_dir, "News Articles")
summaries_dir = os.path.join(base_dir, "Summaries")

pairs = []
max_article_len = 20
max_summary_len = 5

categories = os.listdir(articles_dir)
for cat in categories:
    cat_article_dir = os.path.join(articles_dir, cat)
    cat_summary_dir = os.path.join(summaries_dir, cat)
    if not os.path.isdir(cat_article_dir):
        continue
    for fname in sorted(os.listdir(cat_article_dir)):
        if not fname.endswith(".txt"):
            continue
        article_path = os.path.join(cat_article_dir, fname)
        summary_path = os.path.join(cat_summary_dir, fname)
        if not os.path.exists(summary_path):
            continue
        with open(article_path, "r", encoding="latin-1") as f:
            lines = f.read().strip().split("\n")
        article = " ".join(lines[2:]) if len(lines) > 2 else lines[0]
        article_words = article.split()[:max_article_len]
        if len(article_words) < 5:
            continue
        with open(summary_path, "r", encoding="latin-1") as f:
            summary = f.read().strip()
        summary_words = summary.split()[:max_summary_len]
        if len(summary_words) < 3:
            continue
        pairs.append((" ".join(article_words).lower(), " ".join(summary_words).lower()))

print(f"Loaded {len(pairs)} article-summary pairs")

# Use a smaller subset for faster training
random.seed(42)
random.shuffle(pairs)
pairs = pairs[:100]
split = int(0.85 * len(pairs))
train_pairs = pairs[:split]
test_pairs = pairs[split:]
print(f"Training pairs: {len(train_pairs)}, Test pairs: {len(test_pairs)}")

Using device: cpu
Loaded 2225 article-summary pairs
Training pairs: 85, Test pairs: 15


In [2]:
# Build vocabulary
SPECIAL = ["<PAD>", "<SOS>", "<EOS>", "<UNK>"]
words = set()
for a, b in pairs:
    words.update(a.split())
    words.update(b.split())
vocab = SPECIAL + sorted(words)
stoi = {w: i for i, w in enumerate(vocab)}
itos = {i: w for w, i in stoi.items()}
vocab_size = len(vocab)
print(f"Vocabulary size: {vocab_size}")

def encode(text, add_sos=False, add_eos=False):
    ids = []
    if add_sos:
        ids.append(stoi["<SOS>"])
    for w in text.split():
        ids.append(stoi.get(w, stoi["<UNK>"]))
    if add_eos:
        ids.append(stoi["<EOS>"])
    return torch.tensor(ids, dtype=torch.long)

train_data = [(encode(a), encode(b, True, True)) for a, b in train_pairs]
test_data = [(encode(a), encode(b, True, True)) for a, b in test_pairs]

Vocabulary size: 1228


In [3]:
EMBEDDING_SIZE = 16
HIDDEN_SIZE = 32
VOCAB_SIZE = vocab_size
TARGET_LEN = 5
BATCH_SIZE = 1

In [4]:
shared_embedding = nn.Embedding(VOCAB_SIZE, EMBEDDING_SIZE)

In [5]:
class Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = shared_embedding
        self.rnn = nn.RNN(EMBEDDING_SIZE, HIDDEN_SIZE, batch_first=True)

    def forward(self, x):
        e = self.embedding(x)
        outputs, hidden = self.rnn(e)
        return outputs, hidden

In [6]:
class BahdanauAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.W_s = nn.Linear(HIDDEN_SIZE, HIDDEN_SIZE)
        self.W_h = nn.Linear(HIDDEN_SIZE, HIDDEN_SIZE)
        self.v = nn.Linear(HIDDEN_SIZE, 1)

    def forward(self, decoder_hidden, encoder_outputs, mask=None):
        query = decoder_hidden.unsqueeze(1)
        print(f"Query shape: {query.shape}")  # Debugging line
        energy = torch.tanh(self.W_s(query) + self.W_h(encoder_outputs))  # (batch, seq_len, hidden)
        print(f"Energy shape: {energy.shape}")  # Debugging line
        scores = self.v(energy).squeeze(-1)  # (batch, seq_len)
        print(f"Scores shape: {scores.shape}")  # Debugging line

        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))

        attn_weights = F.softmax(scores, dim=1)  # (batch, seq_len)
        print(f"Attention weights shape: {attn_weights.shape}")  # Debugging line
        context = torch.bmm(attn_weights.unsqueeze(1), encoder_outputs)
        print(f"Context shape: {context.shape}")  # Debugging line
        context = context.squeeze(1)  # (batch, hidden)
        print(f"Context shape after squeeze: {context.shape}")  # Debugging line
        return context, attn_weights

In [7]:
encoder = Encoder()
encoder_input = [stoi["<SOS>"]] + [stoi.get(w, stoi["<UNK>"]) for w in "this is a test article".split()] + [stoi["<EOS>"]]
print(f"Encoder input: {encoder_input}")
encoder_outputs, encoder_hidden = encoder(torch.tensor(encoder_input).unsqueeze(0))
print(f"Encoder outputs shape: {encoder_outputs.shape}, Encoder hidden shape: {encoder_hidden.shape}")

Encoder input: [1, 1106, 579, 78, 3, 3, 2]
Encoder outputs shape: torch.Size([1, 7, 32]), Encoder hidden shape: torch.Size([1, 1, 32])


In [8]:
decoder_hidden = torch.rand(1, 1, 32)  # (num_layers, batch, hidden_size)
print(decoder_hidden)
print(f"Decoder hidden shape: {decoder_hidden.shape}")
print(decoder_hidden.squeeze(0))
print(f"Decoder hidden shape: {decoder_hidden.squeeze(0).shape}")

tensor([[[0.0262, 0.8963, 0.4534, 0.1816, 0.0246, 0.8727, 0.0904, 0.4401,
          0.2462, 0.6284, 0.4898, 0.9996, 0.1229, 0.0992, 0.7813, 0.9952,
          0.5182, 0.5613, 0.1981, 0.1697, 0.0396, 0.4210, 0.8476, 0.1482,
          0.5292, 0.1383, 0.8020, 0.3518, 0.4879, 0.2181, 0.6433, 0.0150]]])
Decoder hidden shape: torch.Size([1, 1, 32])
tensor([[0.0262, 0.8963, 0.4534, 0.1816, 0.0246, 0.8727, 0.0904, 0.4401, 0.2462,
         0.6284, 0.4898, 0.9996, 0.1229, 0.0992, 0.7813, 0.9952, 0.5182, 0.5613,
         0.1981, 0.1697, 0.0396, 0.4210, 0.8476, 0.1482, 0.5292, 0.1383, 0.8020,
         0.3518, 0.4879, 0.2181, 0.6433, 0.0150]])
Decoder hidden shape: torch.Size([1, 32])


In [9]:
bah = BahdanauAttention()
bah.forward(decoder_hidden=decoder_hidden.squeeze(0), encoder_outputs=encoder_outputs)

Query shape: torch.Size([1, 1, 32])
Energy shape: torch.Size([1, 7, 32])
Scores shape: torch.Size([1, 7])
Attention weights shape: torch.Size([1, 7])
Context shape: torch.Size([1, 1, 32])
Context shape after squeeze: torch.Size([1, 32])


(tensor([[ 0.2092, -0.0442,  0.0240,  0.0297,  0.0930, -0.0785,  0.0026,  0.1889,
          -0.3350, -0.0736,  0.0645,  0.1320, -0.1160,  0.0516, -0.3326,  0.3379,
           0.1500, -0.0201, -0.2021,  0.0777, -0.0255, -0.0753,  0.2436, -0.0302,
           0.3814, -0.2163,  0.1022,  0.3159, -0.2546, -0.0332,  0.1053, -0.1875]],
        grad_fn=<SqueezeBackward1>),
 tensor([[0.1347, 0.1528, 0.1218, 0.1266, 0.1570, 0.1485, 0.1587]],
        grad_fn=<SoftmaxBackward0>))

In [10]:
class Decoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = shared_embedding
        self.attention = BahdanauAttention()
        self.rnn = nn.RNN(EMBEDDING_SIZE+HIDDEN_SIZE, HIDDEN_SIZE, batch_first=True)
        self.fc_out = nn.Linear(HIDDEN_SIZE*2+EMBEDDING_SIZE, VOCAB_SIZE)


    def forward_step(self, input_token, hidden, encoder_outputs, mask=None):
        embedded = self.embedding(input_token.view(1, 1))
        context, attn_weights = self.attention(hidden.squeeze(0), encoder_outputs, mask)
        rnn_input = torch.cat((embedded, context.unsqueeze(1)), dim=2)
        output, hidden = self.rnn(rnn_input, hidden)
        output = output.squeeze(1)  # (batch, hidden)

        pred_input = torch.cat((output, context, embedded.squeeze(1)), dim=1)
        prediction = self.fc_out(pred_input)
        return prediction, hidden, attn_weights


    def forward(self, tgt, encoder_last_hidden, encoder_outputs, src_mask=None):
        input_token = tgt[:, 0]
        hidden = encoder_last_hidden

        outputs = torch.zeros(BATCH_SIZE, TARGET_LEN, VOCAB_SIZE).to(device)

        for t in range(1, TARGET_LEN):
            print(itos[input_token.item()])
            prediction, hidden, attn_weights = self.forward_step(input_token, hidden, encoder_outputs)
            outputs[:, t] = prediction
            input_token = tgt[:, t]  # Teacher forcing: use actual next token as next input

        return outputs

In [16]:
src = [stoi["<SOS>"]] + [stoi.get(w, stoi["<UNK>"]) for w in "this news is good".split()] + [stoi["<EOS>"]]
tgt = [stoi["<SOS>"]] + [stoi.get(w, stoi["<UNK>"]) for w in "a good news".split()] + [stoi["<EOS>"]]
encoder = Encoder()
print("torch tensor shape(unsqueeze):", torch.tensor(src).unsqueeze(0).shape)
outputs, hidden = encoder(torch.tensor(src).unsqueeze(0))
src, tgt, outputs.shape, hidden.shape

torch tensor shape(unsqueeze): torch.Size([1, 6])


([1, 1106, 756, 579, 490, 2],
 [1, 78, 490, 756, 2],
 torch.Size([1, 6, 32]),
 torch.Size([1, 1, 32]))

In [12]:
decoder = Decoder()
decoder_input = torch.tensor(tgt).unsqueeze(0)
outputs = decoder(decoder_input, hidden, outputs)
outputs.shape, outputs

<SOS>
Query shape: torch.Size([1, 1, 32])
Energy shape: torch.Size([1, 6, 32])
Scores shape: torch.Size([1, 6])
Attention weights shape: torch.Size([1, 6])
Context shape: torch.Size([1, 1, 32])
Context shape after squeeze: torch.Size([1, 32])
a
Query shape: torch.Size([1, 1, 32])
Energy shape: torch.Size([1, 6, 32])
Scores shape: torch.Size([1, 6])
Attention weights shape: torch.Size([1, 6])
Context shape: torch.Size([1, 1, 32])
Context shape after squeeze: torch.Size([1, 32])
good
Query shape: torch.Size([1, 1, 32])
Energy shape: torch.Size([1, 6, 32])
Scores shape: torch.Size([1, 6])
Attention weights shape: torch.Size([1, 6])
Context shape: torch.Size([1, 1, 32])
Context shape after squeeze: torch.Size([1, 32])
news
Query shape: torch.Size([1, 1, 32])
Energy shape: torch.Size([1, 6, 32])
Scores shape: torch.Size([1, 6])
Attention weights shape: torch.Size([1, 6])
Context shape: torch.Size([1, 1, 32])
Context shape after squeeze: torch.Size([1, 32])


(torch.Size([1, 5, 1228]),
 tensor([[[ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
          [ 0.0963,  0.3785, -0.0774,  ..., -0.1720, -0.1753,  0.4832],
          [ 0.0587,  0.4325,  0.6476,  ...,  0.2317,  0.4421,  0.1659],
          [-0.0201, -0.1929,  0.3385,  ...,  0.5684,  0.3219,  0.4670],
          [-0.3091,  0.4397,  0.3270,  ..., -0.0453,  0.2253,  0.1572]]],
        grad_fn=<CopySlices>))

In [13]:
enc = Encoder().to(device)
dec = Decoder().to(device)
enc, dec

(Encoder(
   (embedding): Embedding(1228, 16)
   (rnn): RNN(16, 32, batch_first=True)
 ),
 Decoder(
   (embedding): Embedding(1228, 16)
   (attention): BahdanauAttention(
     (W_s): Linear(in_features=32, out_features=32, bias=True)
     (W_h): Linear(in_features=32, out_features=32, bias=True)
     (v): Linear(in_features=32, out_features=1, bias=True)
   )
   (rnn): RNN(48, 32, batch_first=True)
   (fc_out): Linear(in_features=80, out_features=1228, bias=True)
 ))

In [14]:
emb_dim = 64
hid = 128
enc = Encoder(vocab_size, emb_dim, hid).to(device)
dec = Decoder(vocab_size, emb_dim, hid).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=stoi["<PAD>"])
opt = optim.Adam(list(enc.parameters()) + list(dec.parameters()), lr=0.001)

TypeError: Encoder.__init__() takes 1 positional argument but 4 were given

In [ ]:
def summarize(sentence, max_len=15):
    enc.eval()
    dec.eval()
    with torch.no_grad():
        src = encode(sentence).to(device)
        h = enc(src)
        inp = torch.tensor([stoi["<SOS>"]], device=device)
        out = []
        for _ in range(max_len):
            logits, h = dec(inp, h)
            pred = logits.argmax(-1).item()
            if itos[pred] == "<EOS>":
                break
            out.append(itos[pred])
            inp = torch.tensor([pred], device=device)
    return " ".join(out)

epochs = 5
for epoch in range(epochs):
    enc.train()
    dec.train()
    total_loss = 0
    for src, tgt in train_data:
        src, tgt = src.to(device), tgt.to(device)
        opt.zero_grad()
        h = enc(src)
        loss = 0
        inp = tgt[0].unsqueeze(0)
        for i in range(1, len(tgt)):
            logits, h = dec(inp, h)
            loss += criterion(logits, tgt[i].unsqueeze(0))
            inp = tgt[i].unsqueeze(0)
        loss.backward()
        opt.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(train_data):.4f}")

Epoch 1/5, Loss: 0.8319
Epoch 2/5, Loss: 0.7066
Epoch 3/5, Loss: 0.6039
Epoch 4/5, Loss: 0.5179
Epoch 5/5, Loss: 0.4463


In [ ]:
# Test on held-out articles
print("=" * 60)
print("TEST ON HELD-OUT BBC NEWS ARTICLES")
print("=" * 60)
for i in range(min(10, len(test_pairs))):
    article, ref_summary = test_pairs[i]
    gen_summary = summarize(article)
    print(f"\nArticle: {article}")
    print(f"Reference: {ref_summary}")
    print(f"Generated: {gen_summary}")

TEST ON HELD-OUT BBC NEWS ARTICLES

Article: oracle has announced it is cutting about 5,000 jobs following the completion of its $10.3bn takeover of its smaller rival peoplesoft last week. the company said it would retain more than 90% of peoplesoft product development and product support staff. the cuts will affect about 9% of the 55,000 staff
Reference: "by retaining the vast majority of peoplesoft technical staff, oracle will have the resources to deliver on the development and
Generated: she said it had received a detailed proposal from mr glazer.but it is not yet

Article: supervolcano, a docu-drama about a volcanic eruption in yellowstone national park in the us, is among the highlights on the bbc one this winter. the â£178m winter schedule also includes the return of doctor who and a drama about angela cannings, who was wrongly convicted of killing two of her
Reference: also on christmas day, john nettles will return in a one-off edition of midsomer murders, while two episodes o